# U5-01 · Gráficos de previsiones y matrices de confusión

*Especialista en Inteligencia Artificial (IFCD107) · Unidad 5*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador; la primera vez tarda unos segundos en cargar las bibliotecas.

> Datos: Fanaee-T, H. y Gama, J. (2014). Event labeling combining ensemble detectors and background knowledge. *Progress in Artificial Intelligence*, 2, 113–127. Bike Sharing, UCI, CC BY 4.0, https://doi.org/10.24432/C5W894
> Datos: Koklu, M. y Ozkan, I. A. (2020). *Computers and Electronics in Agriculture*, 174, 105507. https://doi.org/10.1016/j.compag.2020.105507 · Dry Bean, UCI, CC BY 4.0, https://doi.org/10.24432/C50S4B

## 5.2 · Previsiones de demanda

`s1_previsiones.py`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from bicis import X, y, h, ent, pru

cols = ["hora", "laborable", "temp_c", "humedad", "viento_kmh",
        "tiempo", "mes", "año", "dia_semana", "festivo"]
m = HistGradientBoostingRegressor(loss="poisson", max_iter=500,
                                  learning_rate=0.05, random_state=0)
p = m.fit(X.loc[ent, cols], y[ent]).predict(X.loc[pru, cols])
r = pd.DataFrame({"t": (h.dteday + pd.to_timedelta(h.hr, "h"))[pru],
                  "real": y[pru], "prev": p, "hora": h.hr[pru]})
sem = r[(r.t >= "2012-09-10") & (r.t < "2012-09-17")]
eam_h = r.assign(e=(r.real - r.prev).abs()).groupby("hora").e.mean()
print("EAM por hora: mínimo", round(eam_h.min(), 1), "a las",
      eam_h.idxmin(), "h; máximo", round(eam_h.max(), 1), "a las",
      eam_h.idxmax(), "h")
fig, ej = plt.subplots(1, 3, figsize=(6.2, 1.9), layout="constrained",
                       gridspec_kw={"width_ratios": [2.2, 1, 1]})
ej[0].plot(sem.t, sem.real, lw=0.8, label="real")
ej[0].plot(sem.t, sem.prev, lw=0.8, ls="--", label="previsto")
ej[0].set_xticks(sem.t[sem.t.dt.hour == 0])
ej[0].set_xticklabels([f"{d:%d}" for d in sem.t[sem.t.dt.hour == 0]])
ej[0].set_xlabel("septiembre de 2012"); ej[0].legend(fontsize=6)
ej[0].set_ylabel("alquileres por hora")
ej[1].scatter(r.real, r.prev, s=0.3, alpha=0.3)
ej[1].plot([0, 1000], [0, 1000], color="gray", lw=0.6)
ej[1].set_xlabel("real"); ej[1].set_ylabel("previsto")
ej[2].bar(eam_h.index, eam_h.values)
ej[2].set_xlabel("hora"); ej[2].set_ylabel("EAM")
plt.show()

## 5.3 · Clasificación: la matriz de confusión

`s2_confusion.py`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

j = pd.read_csv("judias.csv").drop_duplicates()
X, y = j.drop(columns="Class"), j.Class.str.title()
X_e, X_p, y_e, y_p = train_test_split(X, y, test_size=0.25,
                                      stratify=y, random_state=0)
p = make_pipeline(StandardScaler(), SVC()).fit(X_e, y_e).predict(X_p)
cl = sorted(y.unique())
cm = confusion_matrix(y_p, p, labels=cl, normalize="true")
print("diagonal (% de cada clase bien clasificada):",
      (np.diag(cm) * 100).round(1))
fig, ej = plt.subplots(figsize=(3.2, 2.8), layout="constrained")
im = ej.imshow(cm * 100, cmap="Blues", vmin=0, vmax=100)
for i in range(7):
    for k in range(7):
        if cm[i, k] >= 0.005:
            ej.text(k, i, f"{cm[i, k] * 100:.0f}", ha="center",
                    va="center", fontsize=6,
                    color="white" if cm[i, k] > 0.5 else "black")
ej.set_xticks(range(7), cl, rotation=45, fontsize=6)
ej.set_yticks(range(7), cl, fontsize=6)
ej.set_xlabel("prevista"); ej.set_ylabel("real")
fig.colorbar(im, label="% de la clase real")
plt.show()

**Prueba.** Cambia la semana de `s1_previsiones.py` por una de diciembre de 2012. ¿Sigue el modelo tan bien a la realidad?